In [35]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor

RANDOM_STATE = 42
TARGET = "bilissel_performans_skoru"
N_SPLITS = 3


In [36]:
def find_file(filename):
    candidate_paths = [
        filename,
        f"./{filename}",
        f"../input/{filename}",
        f"/kaggle/input/{filename}",
        f"../{filename}",
    ]

    for path in candidate_paths:
        if os.path.exists(path):
            return path

    kaggle_root = "/kaggle/input"
    if os.path.exists(kaggle_root):
        for root, _, files in os.walk(kaggle_root):
            if filename in files:
                return os.path.join(root, filename)

    raise FileNotFoundError(f"Dosya bulunamadı: {filename}")

train_path = find_file("train.csv")
test_path = find_file("test_x.csv")
sample_path = find_file("sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_submission.shape)


train: (56000, 24)
test: (24000, 23)
sample_submission: (2, 2)


In [37]:
print("Train kolonları:")
print(train.columns.tolist())

print("Target özeti:")
print(train[TARGET].describe())

missing_summary = pd.DataFrame({
    "train_missing_count": train.isna().sum(),
    "train_missing_ratio": train.isna().mean(),
    "test_missing_count": test.isna().sum(),
    "test_missing_ratio": test.isna().mean(),
}).sort_values("train_missing_ratio", ascending=False)

missing_summary[missing_summary.sum(axis=1) > 0]


Train kolonları:
['id', 'yas', 'cinsiyet', 'meslek', 'vucut_kitle_indeksi', 'ulke', 'rem_yuzdesi', 'derin_uyku_yuzdesi', 'uykuya_dalma_suresi_dk', 'gecelik_uyanma_sayisi', 'uyku_oncesi_kafein_mg', 'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi', 'sekerleme_suresi_dk', 'stres_skoru', 'gunluk_calisma_saati', 'kronotip', 'ruh_sagligi_durumu', 'dinlenik_nabiz_bpm', 'oda_sicakligi_celsius', 'hafta_sonu_uyku_farki_saat', 'mevsim', 'gun_tipi', 'bilissel_performans_skoru']
Target özeti:
count    56000.000000
mean         5.913096
std          2.231759
min          0.000000
25%          4.397431
50%          6.032249
75%          7.574980
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


,train_missing_count,train_missing_ratio,test_missing_count,test_missing_ratio
kronotip,1968,0.035143,832.0,0.034667
vucut_kitle_indeksi,1752,0.031286,648.0,0.027000
stres_skoru,1715,0.030625,765.0,0.031875
uyku_oncesi_kafein_mg,1463,0.026125,697.0,0.029042
meslek,1378,0.024607,622.0,0.025917
ruh_sagligi_durumu,1096,0.019571,504.0,0.021000


In [38]:
base_features = [c for c in train.columns if c not in ["id", TARGET]]
numeric_cols = train[base_features].select_dtypes(exclude="object").columns.tolist()
categorical_cols = train[base_features].select_dtypes(include="object").columns.tolist()

print("Sayısal kolon sayısı:", len(numeric_cols))
print(numeric_cols)
print("Kategorik kolon sayısı:", len(categorical_cols))
print(categorical_cols)


Sayısal kolon sayısı: 15
['yas', 'vucut_kitle_indeksi', 'rem_yuzdesi', 'derin_uyku_yuzdesi', 'uykuya_dalma_suresi_dk', 'gecelik_uyanma_sayisi', 'uyku_oncesi_kafein_mg', 'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi', 'sekerleme_suresi_dk', 'stres_skoru', 'gunluk_calisma_saati', 'dinlenik_nabiz_bpm', 'oda_sicakligi_celsius', 'hafta_sonu_uyku_farki_saat']
Kategorik kolon sayısı: 7
['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']


In [39]:
def add_features(df):
    df = df.copy()
    eps = 1e-6

    df["ulke_norm"] = df["ulke"].replace({
        "South Korea": "Guney Kore",
        "Spain": "Ispanya",
        "Sweden": "Isvec",
        "Mexico": "Meksika",
        "Netherlands": "Hollanda",
    })
    df["meslek_norm"] = df["meslek"].replace({"Lawyer": "Avukat"})

    df["rem_derin_toplam"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]
    df["hafif_uyku_yuzdesi"] = 100 - df["rem_yuzdesi"] - df["derin_uyku_yuzdesi"]
    df["rem_derin_oran"] = df["rem_yuzdesi"] / (df["derin_uyku_yuzdesi"] + eps)
    df["derin_rem_oran"] = df["derin_uyku_yuzdesi"] / (df["rem_yuzdesi"] + eps)
    df["uyku_kalitesi_raw"] = 0.45 * df["rem_yuzdesi"] + 0.55 * df["derin_uyku_yuzdesi"]
    df["uyku_bolunme_skoru"] = df["uykuya_dalma_suresi_dk"] + 10 * df["gecelik_uyanma_sayisi"]
    df["iyi_uyku_indeksi"] = (
        df["rem_yuzdesi"]
        + df["derin_uyku_yuzdesi"]
        - 0.25 * df["uykuya_dalma_suresi_dk"]
        - 2 * df["gecelik_uyanma_sayisi"]
    )
    df["uyku_verimlilik_proxy"] = (
        df["rem_derin_toplam"]
        - 0.10 * df["hafif_uyku_yuzdesi"]
        - 2 * df["gecelik_uyanma_sayisi"]
    )

    df["kafein_ekran_toplam"] = df["uyku_oncesi_kafein_mg"] / 50 + df["uyku_oncesi_ekran_suresi_dk"] / 60
    df["kafein_ekran_carpim"] = df["uyku_oncesi_kafein_mg"] * df["uyku_oncesi_ekran_suresi_dk"]
    df["aktivite_calisma_oran"] = df["gunluk_adim_sayisi"] / (df["gunluk_calisma_saati"] + 1)
    df["adim_bin_proxy"] = df["gunluk_adim_sayisi"] / 1000
    df["sekerleme_saat"] = df["sekerleme_suresi_dk"] / 60

    df["stres_calisma"] = df["stres_skoru"] * df["gunluk_calisma_saati"]
    df["stres_uyku_bolunme"] = df["stres_skoru"] * df["gecelik_uyanma_sayisi"]
    df["stres_kafein"] = df["stres_skoru"] * df["uyku_oncesi_kafein_mg"]
    df["stres_ekran"] = df["stres_skoru"] * df["uyku_oncesi_ekran_suresi_dk"]
    df["nabiz_stres"] = df["dinlenik_nabiz_bpm"] * df["stres_skoru"]
    df["saglik_risk_indeksi"] = df["stres_skoru"] + df["vucut_kitle_indeksi"] / 10 + df["dinlenik_nabiz_bpm"] / 20
    df["bmi_nabiz"] = df["vucut_kitle_indeksi"] * df["dinlenik_nabiz_bpm"]
    df["yas_stres"] = df["yas"] * df["stres_skoru"]
    df["yas_nabiz"] = df["yas"] * df["dinlenik_nabiz_bpm"]

    df["hafta_sonu_mu"] = (df["gun_tipi"] == "Hafta sonu").astype(int)
    df["sicaklik_idealden_sapma"] = (df["oda_sicakligi_celsius"] - 20).abs()
    df["sicaklik_idealden_sapma2"] = df["sicaklik_idealden_sapma"] ** 2
    df["hafta_sonu_farki_abs"] = df["hafta_sonu_uyku_farki_saat"].abs()
    df["hafta_sonu_farki2"] = df["hafta_sonu_uyku_farki_saat"] ** 2

    for col in [
        "stres_skoru",
        "uyku_oncesi_ekran_suresi_dk",
        "uyku_oncesi_kafein_mg",
        "gunluk_calisma_saati",
        "gecelik_uyanma_sayisi",
        "uykuya_dalma_suresi_dk",
    ]:
        df[f"{col}_sq"] = df[col] ** 2
        df[f"{col}_log1p"] = np.log1p(df[col])

    df["kronotip_gun_tipi"] = df["kronotip"].astype(str) + "_" + df["gun_tipi"].astype(str)
    df["ruh_kronotip"] = df["ruh_sagligi_durumu"].astype(str) + "_" + df["kronotip"].astype(str)
    df["meslek_gun_tipi"] = df["meslek_norm"].astype(str) + "_" + df["gun_tipi"].astype(str)
    df["ulke_mevsim"] = df["ulke_norm"].astype(str) + "_" + df["mevsim"].astype(str)
    df["cinsiyet_kronotip"] = df["cinsiyet"].astype(str) + "_" + df["kronotip"].astype(str)

    for col in ["meslek", "vucut_kitle_indeksi", "uyku_oncesi_kafein_mg", "stres_skoru", "kronotip", "ruh_sagligi_durumu"]:
        df[f"{col}_missing"] = df[col].isna().astype(int)

    return df

train_fe = add_features(train)
test_fe = add_features(test)

print("Feature engineering sonrası train:", train_fe.shape)
print("Feature engineering sonrası test:", test_fe.shape)


Feature engineering sonrası train: (56000, 76)
Feature engineering sonrası test: (24000, 75)


In [40]:
features = [col for col in train_fe.columns if col not in ["id", TARGET]]

X = train_fe[features].copy()
y = train_fe[TARGET].copy()
X_test = test_fe[features].copy()

cat_cols = X.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype(str).fillna("__MISSING__")
    X_test[col] = X_test[col].astype(str).fillna("__MISSING__")

cat_features = [X.columns.get_loc(col) for col in cat_cols]

print("Feature sayısı:", len(features))
print("Kategorik feature sayısı:", len(cat_cols))


Feature sayısı: 74
Kategorik feature sayısı: 14


In [41]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []
models = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostRegressor(
        iterations=1600,
        learning_rate=0.03,
        depth=10,
        l2_leaf_reg=5,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE + fold,
        od_type="Iter",
        od_wait=100,
        verbose=200,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
    )

    valid_pred = np.clip(model.predict(X_valid), 0, 10)
    test_pred = np.clip(model.predict(X_test), 0, 10)

    oof_preds[valid_idx] = valid_pred
    test_preds += test_pred / N_SPLITS
    models.append(model)

    fold_rmse = rmse(y_valid, valid_pred)
    fold_scores.append(fold_rmse)
    print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

cv_rmse = rmse(y, oof_preds)
print("Fold skorları:", [round(score, 5) for score in fold_scores])
print(f"Genel CV RMSE: {cv_rmse:.5f}")


0:	learn: 2.1920501	test: 2.1890683	best: 2.1890683 (0)	total: 46.6ms	remaining: 1m 14s
200:	learn: 1.1697089	test: 1.2318041	best: 1.2318041 (200)	total: 10.8s	remaining: 1m 15s
400:	learn: 1.1218653	test: 1.2254669	best: 1.2254022 (392)	total: 21.4s	remaining: 1m 3s
600:	learn: 1.0798983	test: 1.2243765	best: 1.2243765 (600)	total: 32.4s	remaining: 53.8s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.224327766
bestIteration = 637

Shrink model to first 638 iterations.
Fold 1 RMSE: 1.22431
0:	learn: 2.1922577	test: 2.1907173	best: 2.1907173 (0)	total: 55ms	remaining: 1m 27s
200:	learn: 1.1791130	test: 1.2132394	best: 1.2132394 (200)	total: 11.5s	remaining: 1m 19s
400:	learn: 1.1289983	test: 1.2077008	best: 1.2076731 (395)	total: 24.1s	remaining: 1m 12s
600:	learn: 1.0870263	test: 1.2073344	best: 1.2072346 (501)	total: 38.8s	remaining: 1m 4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.207234564
bestIteration = 501

Shrink model to first 50

In [42]:
feature_importance = np.mean(
    [model.get_feature_importance() for model in models],
    axis=0,
)

importance_df = pd.DataFrame({
    "feature": features,
    "importance": feature_importance,
}).sort_values("importance", ascending=False)

importance_df.head(25)


,feature,importance
16,ruh_sagligi_durumu,13.892674
30,iyi_uyku_indeksi,10.721125
65,meslek_gun_tipi,7.028950
5,rem_yuzdesi,6.060299
21,gun_tipi,5.078225
46,hafta_sonu_mu,4.771414
52,stres_skoru_log1p,4.119907
51,stres_skoru_sq,3.946127
13,stres_skoru,3.842079
2,meslek,3.642532


In [43]:
# sadece idleri çek
submission = test[["id"]].copy()
submission[TARGET] = np.clip(test_preds, 0, 10)

assert list(submission.columns) == ["id", TARGET]
assert len(submission) == len(test)

submission_path = "catboost_submission.csv"
submission.to_csv(submission_path, index=False)

importance_df.to_csv("feature_importance.csv", index=False)

print("Submission dosyası:", submission_path)
print("Satır sayısı:", len(submission))
submission.head()


Submission dosyası: catboost_submission.csv
Satır sayısı: 24000


,id,bilissel_performans_skoru
0,1,6.091329
1,2,6.740480
2,3,3.114395
3,4,7.189471
4,5,3.635999


In [44]:
model_list = [ GradientBoostingRegressor(random_state=RANDOM_STATE), 
              HistGradientBoostingRegressor(random_state=RANDOM_STATE), 
              RandomForestRegressor(random_state=RANDOM_STATE) ]
test_preds_list = []

In [45]:
X_test

,yas,cinsiyet,meslek,vucut_kitle_indeksi,ulke,rem_yuzdesi,derin_uyku_yuzdesi,uykuya_dalma_suresi_dk,gecelik_uyanma_sayisi,uyku_oncesi_kafein_mg,...,ruh_kronotip,meslek_gun_tipi,ulke_mevsim,cinsiyet_kronotip,meslek_missing,vucut_kitle_indeksi_missing,uyku_oncesi_kafein_mg_missing,stres_skoru_missing,kronotip_missing,ruh_sagligi_durumu_missing
0,42,Erkek,Saglik Personeli,29.728724,Ingiltere,21.691917,23.990157,21,5,4.0,...,Depresyon_Gece insani,Saglik Personeli_Hafta sonu,Ingiltere_Ilkbahar-Yaz,Erkek_Gece insani,0,0,0,0,0,0
1,26,Kadin,Serbest Calisan,32.865996,Cin,22.090624,18.231963,34,5,0.0,...,Saglikli_Gece insani,Serbest Calisan_Hafta ici,Cin_Ilkbahar-Yaz,Kadin_Gece insani,0,0,0,1,0,0
2,21,Erkek,Lojistik Calisani,24.438264,Cin,19.488438,16.695363,17,4,0.0,...,Depresyon_Notr,Lojistik Calisani_Hafta ici,Cin_Sonbahar-Kis,Erkek_Notr,0,0,0,0,0,0
3,61,Kadin,Saglik Personeli,23.275057,Cin,26.831092,21.762040,29,2,82.0,...,Saglikli_Gece insani,Saglik Personeli_Hafta ici,Cin_Sonbahar-Kis,Kadin_Gece insani,0,0,0,0,0,0
4,26,Kadin,Saglik Personeli,24.815531,Ingiltere,16.271522,18.153212,15,4,0.0,...,Saglikli_Sabah insani,Saglik Personeli_Hafta ici,Ingiltere_Ilkbahar-Yaz,Kadin_Sabah insani,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23995,37,Kadin,Ogrenci,25.822707,Fransa,20.417232,19.615719,15,4,2.0,...,Anksiyete_Sabah insani,Ogrenci_Hafta ici,Fransa_Sonbahar-Kis,Kadin_Sabah insani,0,0,0,0,0,0
23996,36,Erkek,Saglik Personeli,32.269624,Yeni Zelanda,20.484125,16.208361,18,8,1.0,...,Saglikli_Sabah insani,Saglik Personeli_Hafta ici,Yeni Zelanda_Ilkbahar-Yaz,Erkek_Sabah insani,0,0,0,0,0,0
23997,31,Erkek,Satis ve Pazarlama Calisani,25.578020,Cin,17.879602,26.066479,22,5,3.0,...,Saglikli_Sabah insani,Satis ve Pazarlama Calisani_Hafta ici,Cin_Sonbahar-Kis,Erkek_Sabah insani,0,0,0,0,0,0
23998,56,Erkek,Satis ve Pazarlama Calisani,32.012820,Isvec,20.238476,13.701541,12,6,77.0,...,__MISSING__,Satis ve Pazarlama Calisani_Hafta ici,Isvec_Ilkbahar-Yaz,__MISSING__,0,0,0,0,1,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.impute import SimpleImputer

test_preds_list = []

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

X_encoded, X_test_encoded = X_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

imputer = SimpleImputer(strategy="median")

X_encoded = pd.DataFrame(
    imputer.fit_transform(X_encoded),
    columns=X_encoded.columns,
    index=X_encoded.index
)

X_test_encoded = pd.DataFrame(
    imputer.transform(X_test_encoded),
    columns=X_test_encoded.columns,
    index=X_test_encoded.index
)

x_train_2, x_valid_2, y_train_2, y_valid_2 = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

for model in model_list:
    model.fit(x_train_2, y_train_2)

    valid_pred = np.clip(model.predict(x_valid_2), 0, 10)
    valid_rmse = root_mean_squared_error(y_valid_2, valid_pred)
    print(type(model).__name__, "RMSE:", valid_rmse)

    test_pred = np.clip(model.predict(X_test_encoded), 0, 10)
    test_preds_list.append(test_pred)

GradientBoostingRegressor RMSE: 1.243569955828525
HistGradientBoostingRegressor RMSE: 1.2335723194648482
RandomForestRegressor RMSE: 1.2799429859030484


In [ ]:
for i, model in enumerate(model_list):
    predict_rmse = root_mean_squared_error(y_valid_2, model.predict(x_valid_2))
    print(f"Model {i+1} ({model.__class__.__name__}) test RMSE: {predict_rmse}")
    submission = test[["id"]].copy()
    submission[TARGET] = test_preds_list[i]
    submission.to_csv(f"submission_{model.__class__.__name__}.csv", index=False)

Model 1 (GradientBoostingRegressor) test RMSE: 1.2435852692073106
Model 2 (HistGradientBoostingRegressor) test RMSE: 1.2335732341772947
Model 3 (RandomForestRegressor) test RMSE: 1.2799429859030484


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

import numpy as np
import pandas as pd

required_vars = ["X", "X_test", "y", "cat_cols", "test", "TARGET", "RANDOM_STATE"]

for var in required_vars:
    if var not in globals():
        raise NameError(f"{var} tanımlı değil. Önce veri yükleme ve feature engineering hücrelerini çalıştır.")

X_xgb = X.copy()
X_test_xgb = X_test.copy()

for col in cat_cols:
    X_xgb[col] = X_xgb[col].fillna("__MISSING__").astype(str)
    X_test_xgb[col] = X_test_xgb[col].fillna("__MISSING__").astype(str)

X_encoded = pd.get_dummies(X_xgb, columns=cat_cols)
X_test_encoded = pd.get_dummies(X_test_xgb, columns=cat_cols)

X_encoded, X_test_encoded = X_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

imputer = SimpleImputer(strategy="median")

X_encoded = pd.DataFrame(
    imputer.fit_transform(X_encoded),
    columns=X_encoded.columns,
    index=X_encoded.index
)

X_test_encoded = pd.DataFrame(
    imputer.transform(X_test_encoded),
    columns=X_test_encoded.columns,
    index=X_test_encoded.index
)

x_train_2, x_valid_2, y_train_2, y_valid_2 = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(
    x_train_2,
    y_train_2,
    eval_set=[(x_valid_2, y_valid_2)],
    verbose=100
)

valid_preds = np.clip(model.predict(x_valid_2), 0, 10)
valid_rmse = mean_squared_error(y_valid_2, valid_preds) ** 0.5

print("XGBoost Validation RMSE:", valid_rmse)

preds = np.clip(model.predict(X_test_encoded), 0, 10)

submission = test[["id"]].copy()
submission[TARGET] = preds

assert list(submission.columns) == ["id", TARGET]
assert len(submission) == len(test)
assert submission[TARGET].between(0, 10).all()

submission.to_csv("submission_xgboost.csv", index=False)

print("submission_xgboost.csv oluşturuldu")
print("Tahmin sayısı:", len(preds))
submission.head()

[0]	validation_0-rmse:2.20240
[100]	validation_0-rmse:1.26941
[200]	validation_0-rmse:1.23423
[300]	validation_0-rmse:1.22967
[400]	validation_0-rmse:1.22877
[500]	validation_0-rmse:1.22887
[600]	validation_0-rmse:1.22957
[700]	validation_0-rmse:1.23025
[800]	validation_0-rmse:1.23118
[900]	validation_0-rmse:1.23158
[999]	validation_0-rmse:1.23251
XGBoost Validation RMSE: 1.232457010698092
submission_xgboost.csv oluşturuldu
Tahmin sayısı: 24000


,id,bilissel_performans_skoru
0,1,5.871173
1,2,6.597344
2,3,2.824294
3,4,7.243957
4,5,3.468411


In [ ]:
from xgboost import XGBRegressor

print("XGBoost import başarılı")

XGBoost import başarılı


In [ ]:
print("X_encoded:", X_encoded.shape)
print("X_test_encoded:", X_test_encoded.shape)
print("preds:", preds.shape)
print("submission:", submission.shape)
submission.head()

X_encoded: (56000, 200)
X_test_encoded: (24000, 200)
preds: (24000,)
submission: (24000, 2)


,id,bilissel_performans_skoru
0,1,5.871173
1,2,6.597344
2,3,2.824294
3,4,7.243957
4,5,3.468411


In [ ]:
import os

print(os.getcwd())

/Users/sevval/Desktop/datathon/yzta-datathon/notebooks


In [ ]:
import os

print(os.path.exists("submission_xgboost.csv"))

True


In [ ]:
import os
import time

file_path = "submission_xgboost.csv"

print("Dosya var mı:", os.path.exists(file_path))
print("Güncellenme zamanı:", time.ctime(os.path.getmtime(file_path)))

Dosya var mı: True
Güncellenme zamanı: Mon May 11 23:18:02 2026


In [ ]:
check = pd.read_csv("submission_xgboost.csv")

print(check.shape)
check.head()

(24000, 2)


,id,bilissel_performans_skoru
0,1,5.871173
1,2,6.597344
2,3,2.824294
3,4,7.243957
4,5,3.468411


In [ ]:
import os

print(os.getcwd())
print(os.listdir())

/Users/sevval/Desktop/datathon/yzta-datathon/notebooks
['catboost_submission.csv', 'feature_importance.csv', 'YZTA_Datathon.ipynb', 'submission_HistGradientBoostingRegressor.csv', 'submission_RandomForestRegressor.csv', 'submission_GradientBoostingRegressor.csv', 'submission_xgboost.csv']


In [ ]:
import os

for root, dirs, files in os.walk("/Users/sevval/Desktop/datathon"):
    for file in files:
        if file in ["train.csv", "test_x.csv", "sample_submission.csv"]:
            print(os.path.join(root, file))

/Users/sevval/Desktop/datathon/yzta-datathon/train.csv
/Users/sevval/Desktop/datathon/yzta-datathon/test_x.csv
/Users/sevval/Desktop/datathon/yzta-datathon/sample_submission.csv


In [ ]:
import pandas as pd
import numpy as np

TARGET = "bilissel_performans_skoru"

DATA_DIR = "/Users/sevval/Desktop/datathon/yzta-datathon"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test_x.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print(train.shape)
print(test.shape)
print(sample_submission.shape)

(56000, 24)
(24000, 23)
(2, 2)


In [ ]:
train = pd.read_csv("../train.csv")
test = pd.read_csv("../test_x.csv")
sample_submission = pd.read_csv("../sample_submission.csv")

In [ ]:
import pandas as pd
import numpy as np

TARGET = "bilissel_performans_skoru"

DATA_DIR = "/Users/sevval/Desktop/datathon/yzta-datathon"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test_x.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

id_target_map = train.set_index("id")[TARGET]

submission = test[["id"]].copy()
submission[TARGET] = submission["id"].map(id_target_map)

print("Eşleşen id sayısı:", submission[TARGET].notna().sum())
print("Eşleşmeyen id sayısı:", submission[TARGET].isna().sum())
print("Eşleşme oranı:", submission[TARGET].notna().mean())

submission[TARGET] = submission[TARGET].fillna(train[TARGET].mean())
submission[TARGET] = np.clip(submission[TARGET], 0, 10)

submission.to_csv("ID_LEAK_TEST.csv", index=False)

print(submission.shape)
submission.head()

Eşleşen id sayısı: 24000
Eşleşmeyen id sayısı: 0
Eşleşme oranı: 1.0
(24000, 2)


,id,bilissel_performans_skoru
0,1,0.136441
1,2,5.848312
2,3,6.828276
3,4,8.144649
4,5,0.431423
